In [1]:
import numpy as np
import pandas as pd
import dill 
from tqdm import tqdm
import time
from scipy.stats import qmc
import random
import scipy.stats as stats
from scipy.stats import multivariate_normal

from Forward_model.generate_IC_fixed_grid import get_generate_IC_fixed_grid
from Forward_model.solution_using_FD import get_solution_using_FD
from Forward_model.linear_interpolation import interpolate_linear,interpolate_linear_fill_value
import os

from sklearn.gaussian_process.kernels import Matern
from Forward_model.cubic_spline_interpolation import interpolate_cubicSpline

In [2]:
import matplotlib.pyplot as plt

In [3]:
Ts=-44.1
ws=64
Tb=-1.4
wb=0
m=11
alpha=2.4634
beta=0
velocity=[]
dt_approx=2**-4
dz_approx=2**2

EDML=pd.read_excel('Forward_model/paper_site_densities.xlsx')
borehole3000_densityProf=list(EDML['den_EDML'].dropna())
borehole3000_depths= list(EDML['dep_EDML'].dropna())


In [4]:
ics={}
with open('Forward_model/IC_ws64_wb0_tb14_H2782_a24634_ic.pkl', 'rb') as f:
    ics = dill.load(f)

In [5]:
def select_ICs(ics,Ts_new):
    ic=ics['%.3f'%(np.round(Ts_new,3))]
       
    return ic

In [6]:
ic=select_ICs(ics,Ts)
max_time=1000-1
time_grid=int(max_time/dt_approx)
time_req=np.linspace(0,max_time,time_grid)
max_depth=2782

In [7]:
max_time=500-1
time_grid=int(max_time/dt_approx)
time_req=np.linspace(0,max_time,time_grid)+500
max_depth=2782

In [8]:
no_of_profiles=1000
starting_prof=0
T_PolarAmplification06 = np.genfromtxt(fname='Data/case_study/raw_data/EDML_T_PolarAmplification0.beta06.txt')
T_PolarAmplification16 = np.genfromtxt(fname='Data/case_study/raw_data/EDML_T_PolarAmplification1.beta06.txt')
T_PolarAmplification26 = np.genfromtxt(fname='Data/case_study/raw_data/EDML_T_PolarAmplification2.beta06.txt')
T_PolarAmplification36 = np.genfromtxt(fname='Data/case_study/raw_data/EDML_T_PolarAmplification3.beta06.txt')
time_points_500=[]
for i in range (500,1000):#maxtime
    time_points_500.append(i)
velocity=[]

In [9]:
rsignal_1=[]
for k in range(starting_prof,no_of_profiles):#no. of profiles
    TPA_data=[]
    for i in range (500,1000):#maxtime
        TPA_data.append(T_PolarAmplification06[i][k])
    top_boundary=interpolate_linear(x_ref=time_points_500,y_ref=TPA_data,x_req=time_req)
    rsignal_1.append(np.array(top_boundary))   

In [10]:
rsignal_2=[]
for k in range(starting_prof,no_of_profiles):#no. of profiles
    TPA_data=[]
    for i in range (500,1000):#maxtime
        TPA_data.append(T_PolarAmplification16[i][k])
    top_boundary=interpolate_linear(x_ref=time_points_500,y_ref=TPA_data,x_req=time_req)
    rsignal_2.append(np.array(top_boundary))     

In [11]:
rsignal_3=[]
for k in range(starting_prof,no_of_profiles):#no. of profiles
    TPA_data=[]
    for i in range (500,1000):#maxtime
        TPA_data.append(T_PolarAmplification26[i][k])
    top_boundary=interpolate_linear(x_ref=time_points_500,y_ref=TPA_data,x_req=time_req)
    rsignal_3.append(np.array(top_boundary))     

In [12]:
rsignal_4=[]
for k in range(starting_prof,no_of_profiles):#no. of profiles
    TPA_data=[]
    for i in range (500,1000):#maxtime
        TPA_data.append(T_PolarAmplification36[i][k])
    top_boundary=interpolate_linear(x_ref=time_points_500,y_ref=TPA_data,x_req=time_req)
    rsignal_4.append(np.array(top_boundary))     

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.gaussian_process.kernels import Matern



def kernel_based_model(t, t_i, alpha_i, length_scale, nu):

    t = np.atleast_1d(t).reshape(-1, 1)
    t_i = np.array(t_i).reshape(-1, 1)
    alpha_i = np.array(alpha_i).reshape(-1)

    kernel = Matern(length_scale=length_scale, nu=nu)
    
    K = kernel(t, t_i)

    st = K @ alpha_i 
    
    return st

def generate_surface_temperature(alpha_list,time_list,time_steps,pom,scale_value):
    
    t_i = time_list 
    
    alpha_i = alpha_list

    t_eval =time_steps
    st_values = kernel_based_model(t_eval, t_i, alpha_i, length_scale=scale_value, nu=np.inf)+pom
      

    return  st_values.flatten()

    
def loss_fn1(weights):
    y_hat = generate_surface_temperature(weights,t_init,t,pom,scale_value)
    return np.mean((y - y_hat)**2)

case_TPA_data_500=get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                        borehole3000_depths, borehole3000_densityProf,
                                        velocity,ic,-44.1,Tb,alpha,ws, wb,m)

In [15]:
from multiprocessing import Pool
import numpy as np
from scipy.optimize import minimize

t = case_TPA_data_500.times

all_no_of_kernels = [40]
all_scale_values = [20]
no_of_kernels = all_no_of_kernels[0]
scale_value = all_scale_values[0]
t_init = np.linspace(0, 499, no_of_kernels)

pom = 0
def process_signal(args):
    i, signals = args
    y = signals + 44.1

    def loss_fn1_local(w):
        y_hat = generate_surface_temperature(w, t_init, t, pom, scale_value)
        return np.mean((y - y_hat)**2)

    w0 = np.zeros(no_of_kernels)
    res = minimize(
        loss_fn1_local,
        w0,
        method='L-BFGS-B',
        options={'maxiter': 50000, 'disp': True}
    )

    opt_weights = res.x
    y_hat = generate_surface_temperature(opt_weights, t_init, t, pom, scale_value)
    # print("y", y)

    return (
        f"signal1_diff{i}",
        np.abs(y - y_hat),
        f"signal1_weights{i}",
        opt_weights
    )


In [16]:
if __name__ == "__main__":
    dict_signal_1 = {}

    inputs = list(enumerate(rsignal_1, start=1))
    
    with Pool(122) as pool:
        results = pool.map(process_signal, inputs)

In [17]:
for diff_key, diff_val, weight_key, weight_val in results:
    dict_signal_1[diff_key] = diff_val
    dict_signal_1[weight_key] = weight_val

In [19]:
if __name__ == "__main__":
    dict_signal_2 = {}

    inputs = list(enumerate(rsignal_2, start=1))
    
    with Pool(200) as pool:
        results2 = pool.map(process_signal, inputs)

In [21]:

for diff_key, diff_val, weight_key, weight_val in results2:
    dict_signal_2[diff_key] = diff_val
    dict_signal_2[weight_key] = weight_val

In [23]:

dict_signal_3 = {}

inputs = list(enumerate(rsignal_3, start=1))

with Pool(200) as pool:
    results = pool.map(process_signal, inputs)



In [24]:
for diff_key, diff_val, weight_key, weight_val in results:
    dict_signal_3[diff_key] = diff_val
    dict_signal_3[weight_key] = weight_val

In [26]:

dict_signal_4 = {}

inputs = list(enumerate(rsignal_4, start=1))

with Pool(200) as pool:
    results = pool.map(process_signal, inputs)


In [27]:

for diff_key, diff_val, weight_key, weight_val in results:
    dict_signal_4[diff_key] = diff_val
    dict_signal_4[weight_key] = weight_val

In [33]:

with open('Data/case_study/approximation_uncertainty/dict_signal_1.pkl', 'rb') as f:
    loaded_dict1=dill.load(f)

with open('Data/case_study/approximation_uncertainty/dict_signal_2.pkl', 'rb') as f:
    loaded_dict2=dill.load(f)

with open('Data/case_study/approximation_uncertainty/dict_signal_3.pkl', 'rb') as f:
    loaded_dict3=dill.load(f)
with open('Data/case_study/approximation_uncertainty/dict_signal_4.pkl', 'rb') as f:
    loaded_dict4=dill.load(f)


In [34]:
all_dicts=[loaded_dict1,loaded_dict2,loaded_dict3,loaded_dict4]


all_no_of_kernels = [40]
all_scale_values = [20]
no_of_kernels = all_no_of_kernels[0]
scale_value = all_scale_values[0]
t_init = np.linspace(0, 499, no_of_kernels)

t = case_TPA_data_500.times

In [35]:
case_TPA_data_500_06=[]
for k in range(starting_prof,no_of_profiles):
    
    top_boundary=rsignal_1[k]
    bottom_boundary=Tb
    ic=select_ICs(ics,generate_surface_temperature(all_dicts[0][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)[0])
    case_TPA_data_500_06.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                        borehole3000_depths, borehole3000_densityProf,
                                        velocity,ic,top_boundary,Tb,alpha,ws, wb,m))
    

In [36]:
case_TPA_data_500_16=[]
for k in range(starting_prof,no_of_profiles):
    top_boundary=rsignal_2[k]
    bottom_boundary=Tb
    ic=select_ICs(ics,generate_surface_temperature(all_dicts[1][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)[0])
    case_TPA_data_500_16.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                        borehole3000_depths, borehole3000_densityProf,
                                        velocity,ic,top_boundary,Tb,alpha,ws, wb,m))

In [37]:
case_TPA_data_500_26=[]
for k in range(starting_prof,no_of_profiles):
    top_boundary=rsignal_3[k]
    bottom_boundary=Tb
    ic=select_ICs(ics,generate_surface_temperature(all_dicts[2][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)[0])
    case_TPA_data_500_26.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                        borehole3000_depths, borehole3000_densityProf,
                                        velocity,ic,top_boundary,Tb,alpha,ws, wb,m))

In [38]:
case_TPA_data_500_36=[]
for k in range(starting_prof,no_of_profiles):
    top_boundary=rsignal_4[k]
    bottom_boundary=Tb
    ic=select_ICs(ics,generate_surface_temperature(all_dicts[3][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)[0])
    case_TPA_data_500_36.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                        borehole3000_depths, borehole3000_densityProf,
                                        velocity,ic,top_boundary,Tb,alpha,ws, wb,m))

In [39]:
all_fwds_40kic=[]
all_fwds_40kic.append(case_TPA_data_500_06)
all_fwds_40kic.append(case_TPA_data_500_16)
all_fwds_40kic.append(case_TPA_data_500_26)
all_fwds_40kic.append(case_TPA_data_500_36)

In [40]:
case_TPA_data_500_06_kernel40_20=[]
case_TPA_data_500_16_kernel40_20=[]
case_TPA_data_500_26_kernel40_20=[]
case_TPA_data_500_36_kernel40_20=[]
for k in range(starting_prof,no_of_profiles):
    Ts_kernel40_20=generate_surface_temperature(all_dicts[0][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)
    top_boundary=Ts_kernel40_20
    ic=select_ICs(ics,top_boundary[0])
    case_TPA_data_500_06_kernel40_20.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                            borehole3000_depths, borehole3000_densityProf,
                                            velocity,ic,top_boundary,Tb,alpha,ws, wb,m))

    Ts_kernel40_20=generate_surface_temperature(all_dicts[1][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)
    top_boundary=Ts_kernel40_20
    ic=select_ICs(ics,top_boundary[0])
    case_TPA_data_500_16_kernel40_20.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                            borehole3000_depths, borehole3000_densityProf,
                                            velocity,ic,top_boundary,Tb,alpha,ws, wb,m))
    Ts_kernel40_20=generate_surface_temperature(all_dicts[2][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)
    top_boundary=Ts_kernel40_20
    ic=select_ICs(ics,top_boundary[0])
    case_TPA_data_500_26_kernel40_20.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                            borehole3000_depths, borehole3000_densityProf,
                                            velocity,ic,top_boundary,Tb,alpha,ws, wb,m))
    Ts_kernel40_20=generate_surface_temperature(all_dicts[3][f"signal1_weights{k+1}"],np.linspace(0, 499, 40) ,t,-44.1,scale_value)
    top_boundary=Ts_kernel40_20
    ic=select_ICs(ics,top_boundary[0])
    case_TPA_data_500_36_kernel40_20.append(get_solution_using_FD(max_depth, max_time, dz_approx, dt_approx, 
                                            borehole3000_depths, borehole3000_densityProf,
                                            velocity,ic,top_boundary,Tb,alpha,ws, wb,m))


In [41]:
all_fwds_40=[case_TPA_data_500_06_kernel40_20,case_TPA_data_500_16_kernel40_20,case_TPA_data_500_26_kernel40_20,case_TPA_data_500_36_kernel40_20]


In [42]:
all_diffs = []

for i in range(4):
    diffs_i = []
    for j in range(1000):
        diff =  all_fwds_40[i][j].T[:, -1] -  all_fwds_40kic[i][j].T[:, -1]
       

        diffs_i.append(diff)

    all_diffs.append(diffs_i)

In [44]:
all_diffs_loaded=np.load("Data/case_study/approximation_uncertainty/all_diffs.npy")

In [46]:
all_diffs_loaded.shape

(4, 1000, 695)

In [79]:
rms = np.sqrt(np.mean(all_diffs_loaded**2, axis=(0, 1)))
mean_val=np.mean(np.abs(all_diffs_loaded), axis=1)